# Análisis bivariado — ClassicModels

**Persona 3** · Partes 4b, 5 y ensamblaje final

Este notebook usa la misma conexión y carga de tablas que la Persona 2.

## Celda de conexión a la BD

**Qué hace:** abre una conexión a MySQL (`classicmodels`) y carga cada tabla principal en un DataFrame de pandas.

**Cómo funciona:**
1. `mysql.connector.connect(...)` se conecta al contenedor Docker en `127.0.0.1:3307` (mapeado en `docker-compose.yml` porque el 3306 local ya estaba ocupado).
2. `pd.read_sql(...)` ejecuta un `SELECT *` por tabla y guarda el resultado en memoria.
3. Al final se cierra la conexión para no dejar el motor ocupado.

**Para qué sirve:** tener los datos listos en Python para graficar con seaborn/matplotlib (análisis bivariado) sin volver a escribir SQL en cada celda.

**Requisito previo:** la BD debe estar corriendo (`docker compose up -d`).

In [12]:
import mysql.connector
import pandas as pd

# IMPORTANTE: port=3307 (ver docker-compose.yml). Sin esto Python usa 3306 y falla.
connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3307,
    user="root",
    password="root",
    database="classicmodels",
)

print("Conexión OK →", connection.server_host)

df_customers = pd.read_sql("SELECT * FROM customers", connection)
df_employees = pd.read_sql("SELECT * FROM employees", connection)
df_offices = pd.read_sql("SELECT * FROM offices", connection)
df_orders = pd.read_sql("SELECT * FROM orders", connection)
df_orderdetails = pd.read_sql("SELECT * FROM orderdetails", connection)
df_payments = pd.read_sql("SELECT * FROM payments", connection)
df_products = pd.read_sql("SELECT * FROM products", connection)
df_productlines = pd.read_sql("SELECT * FROM productlines", connection)

print("Tablas cargadas:")
for nombre, df in [
    ("customers", df_customers),
    ("employees", df_employees),
    ("offices", df_offices),
    ("orders", df_orders),
    ("orderdetails", df_orderdetails),
    ("payments", df_payments),
    ("products", df_products),
    ("productlines", df_productlines),
]:
    print(f"  - {nombre}: {df.shape[0]} filas × {df.shape[1]} columnas")

connection.close()
print("Conexión cerrada")

Conexión OK → 127.0.0.1
Tablas cargadas:
  - customers: 122 filas × 13 columnas
  - employees: 23 filas × 8 columnas
  - offices: 7 filas × 9 columnas
  - orders: 326 filas × 7 columnas
  - orderdetails: 2996 filas × 5 columnas
  - payments: 273 filas × 4 columnas
  - products: 110 filas × 9 columnas
  - productlines: 7 filas × 4 columnas
Conexión cerrada


/var/folders/26/ln42rh4x0jq01r21hbkqvx3m0000gn/T/ipykernel_6248/3533005456.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customers = pd.read_sql("SELECT * FROM customers", connection)
/var/folders/26/ln42rh4x0jq01r21hbkqvx3m0000gn/T/ipykernel_6248/3533005456.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_employees = pd.read_sql("SELECT * FROM employees", connection)
/var/folders/26/ln42rh4x0jq01r21hbkqvx3m0000gn/T/ipykernel_6248/3533005456.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_offices = 

En customers estoy viendo que puedo agrupar con country y creditLimit. puede que haya una relación como que quizá le dan más crédito a personas de cierto país mientras que a otros no.

En employee puedo ver que quizá algunos empleados tengan asignados más clientes que otros y si es así. 

En office la verdad no veo mucha cosa interesante más que country.

En order podría ver cuántas órdenes hay por cliente y a qué cliente se le demora menos en hacer el ship. por ejemplo puede que haya algún cliente que repetitivamente haya menos tiempo entre orderdate y shippeddate y puede que estos clientes sean de un país en específico tremendooo.

En orderdetails podría unirlo con order yd arme cuenta qué clientes compran más en terminos de priceEach*quantityOrdered. Y quién sabe, si le meto mucha lógica podría saber si hay alguna relación entre qué tanto compra el cliente con qué tan rápido se le hace el envío.

En payments la verdad no sabría qué hacer quizá podría también darme cuenta si entre más amount tienen en sus payments, más rápido les llega el envío

En products podría preguntarme si la quantityInStock es proporcional a cómo se vende ese producto.

En product line podria preguntarme cuál es la linea a la que más le compran